#Initialization


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

In [0]:
df.display()

In [0]:
df.printSchema()

#Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Date Casting

In [0]:
%sql
select sls_due_dt,sls_order_dt, sls_ship_dt
from workspace.bronze.crm_sales_details
where sls_due_dt = 0 or sls_order_dt = 0 or sls_ship_dt = 0

In [0]:
date_cols = ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]

for col_name in date_cols:
    df = df.withColumn(
        col_name,
        F.when(F.length(col_name) == 8, F.to_date(col_name, "yyyyMMdd"))
        .when(F.length(col_name) == 10, F.to_date(col_name, "yyyy-MM-dd"))
        .otherwise(None)
    )

In [0]:
df.printSchema()

In [0]:
df.filter(
    (F.col("sls_due_dt").isNull()) |
    (F.col("sls_order_dt").isNull()) |
    (F.col("sls_ship_dt").isNull())
).select("sls_due_dt", "sls_order_dt", "sls_ship_dt").show() 

##Price Correction

In [0]:

df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)


## Renaming Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe


In [0]:
df.limit(10).display()

#Writing into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

## Sanity checks of silver table


In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10